# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

Problem statement: To build a tool that takes a technical question, and responds with an explanation

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr # oh yeah!
from IPython.display import Markdown, display
from bs4 import BeautifulSoup
import requests

In [2]:
# constants
models = ["openai/gpt-4o-mini", "openai/gpt-5-nano", "openai/gpt-4.1-mini", "anthropic/claude-3-5-sonnet", "ollama/llama3.2"]

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [4]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-


In [5]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

# openai = OpenAI()

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)

In [6]:
system_message = """
You are a senior technical and helpful assistant.

You explain complex technical concepts clearly and accurately.

If a question is asked about religious matters other than Halleluyah Challenge, respond with: "I'm not able to answer that question."

If the question is about Halleluyah Challenge, you MUST use tool calling.

Available tools:
- fetch_website_links(url)
- fetch_website_contents(url)

Base website:
https://www.hallelujahchallengelive.com

You may call fetch_website_links first to discover relevant pages. Then call fetch_website_contents with the correct page URL.
Use the tool results to answer the question.

"""

In [7]:

# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [8]:
def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "fetch_website_links",
            "description": "Return the links on the website at the given base url",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The base url of the website"
                    }
                },
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_website_contents",
            "description": "Return the title and contents of the website at the given url and use this to answer the question",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The full url of the page"
                    }
                },
                "required": ["url"]
            }
        }
    }
]

In [10]:
def build_payload(question: str, model: str) -> str:
    payload = {}
    if model in models:
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": question}
            ],
            "temperature": 0.3,
            "tools": tools,
            "tool_choice": "auto"
        }
    else:
        raise ValueError(f"Invalid model: {model}")

    return payload

In [11]:
def build_stream_payload(question: str, model: str) -> str:
    payload = {}
    if model in models:
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": question}
            ],
            "temperature": 0.3,
            "tools": tools,
            "tool_choice": "auto",
            "stream": True
        }
    else:
        raise ValueError(f"Invalid model: {model}")

    return payload

In [12]:
def printmd(string):
    display(Markdown(string))

In [13]:
def fetch_relevant_website_links(url : str):
    return fetch_website_links(url)

In [15]:
def invoke_model(question, model) -> str:
    payload = build_payload(question, model)

    while True:
        response = openrouter.chat.completions.create(**payload)
        message = response.choices[0].message

        # If model returns normal content → we're done
        if message.content:
            return message.content

        # If tool call requested
        if message.tool_calls:
            for tool_call in message.tool_calls:
                function_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)

                if function_name == "fetch_website_links":
                    result = fetch_website_links(**arguments)

                elif function_name == "fetch_website_contents":
                    result = fetch_website_contents(**arguments)

                # Append assistant tool call
                payload["messages"].append({
                    "role": "assistant",
                    "tool_calls": [{
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": function_name,
                            "arguments": tool_call.function.arguments
                        }
                    }]
                })

                # Append tool result
                payload["messages"].append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

        else:
            # Safety fallback
            return "No response generated."

In [16]:
def invoke_stream_model(question, model):
    yield "🤖 Thinking...\n\n"

    # Step 1: First call WITHOUT streaming
    payload = build_stream_payload(question, model)
    payload["stream"] = False

    response = openrouter.chat.completions.create(**payload)
    message = response.choices[0].message

    # Step 2: Handle tool calls in loop
    while message.tool_calls:
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if function_name == "fetch_website_links":
                result = fetch_website_links(**arguments)
            elif function_name == "fetch_website_contents":
                result = fetch_website_contents(**arguments)

            payload["messages"].append({
                "role": "assistant",
                "tool_calls": [{
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": function_name,
                        "arguments": tool_call.function.arguments
                    }
                }]
            })

            payload["messages"].append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        response = openrouter.chat.completions.create(**payload)
        message = response.choices[0].message

    # Step 3: Stream final answer only
    payload["stream"] = True
    stream = openrouter.chat.completions.create(**payload)

    collected = ""
    for chunk in stream:
        # Skip empty chunks
        if not chunk.choices:
            continue

        choice = chunk.choices[0]

        # Some chunks may not have delta
        if not hasattr(choice, "delta") or choice.delta is None:
            continue

        delta = choice.delta

        if delta.content:
            collected += delta.content
            yield collected

In [17]:

# response = invoke_stream_model(question, models[0])
# printmd(response)

def router(question, model, mode):
    if mode == "Streaming":
        # Yield from streaming generator
        yield from invoke_stream_model(question, model)
    else:
        # Wrap normal response in generator
        result = invoke_model(question, model)
        yield result

In [19]:
message_input = gr.Textbox(label="Your message:", info="Enter a question for the LLM", lines=7)
model_selector = gr.Dropdown(models, label="Select model", value=models[0])
message_output = gr.Markdown(label="Response:")
mode_selector = gr.Radio(["Normal", "Streaming"],  value="Streaming", label="Response Mode")

view = gr.Interface(
    fn=router,
    title="LLMs", 
    inputs=[message_input, model_selector, mode_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", models[0]],
            ["Explain the Transformer architecture to an aspiring AI engineer", models[-1]]
        ], 
    flagging_mode="never",
)
view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
